In [ ]:
import json
import os
import shutil
from pathlib import Path
import random

# ---------------- CLASS MAPPING ----------------
class_map = {
    "accident": 0,
    "non_accident": 1
}

# ---------------- CONVERT LABELME TO YOLO ----------------
def convert_labelme_to_yolo(json_file, labels_output_dir, images_output_dir):

    with open(json_file, 'r') as f:
        data = json.load(f)

    image_width = data['imageWidth']
    image_height = data['imageHeight']

    image_name = data['imagePath']

    txt_name = Path(image_name).stem + ".txt"
    txt_path = os.path.join(labels_output_dir, txt_name)

    with open(txt_path, 'w') as txt_file:

        for shape in data['shapes']:

            label = shape['label']

            if label not in class_map:
                continue

            class_id = class_map[label]

            points = shape['points']

            # Bounding box points
            x1 = points[0][0]
            y1 = points[0][1]

            x2 = points[1][0]
            y2 = points[1][1]

            # Convert to YOLO format
            x_center = ((x1 + x2) / 2) / image_width
            y_center = ((y1 + y2) / 2) / image_height

            width = abs(x2 - x1) / image_width
            height = abs(y2 - y1) / image_height

            txt_file.write(
                f"{class_id} {x_center} {y_center} {width} {height}\n"
            )

    # Copy image
    src_image = os.path.join(os.path.dirname(json_file), image_name)

    dst_image = os.path.join(images_output_dir, image_name)

    if os.path.exists(src_image):
        shutil.copy(src_image, dst_image)

# ---------------- SPLIT DATASET ----------------
def split_dataset(images_dir, labels_dir, output_dir):

    image_files = [f for f in os.listdir(images_dir)
                   if f.endswith(".jpg")]

    random.shuffle(image_files)

    train_split = int(len(image_files) * 0.7)
    val_split = int(len(image_files) * 0.9)

    splits = {
        "train": image_files[:train_split],
        "val": image_files[train_split:val_split],
        "test": image_files[val_split:]
    }

    for split_name, files in splits.items():

        os.makedirs(
            os.path.join(output_dir, "images", split_name),
            exist_ok=True
        )

        os.makedirs(
            os.path.join(output_dir, "labels", split_name),
            exist_ok=True
        )

        for image_file in files:

            txt_file = Path(image_file).stem + ".txt"

            shutil.copy(
                os.path.join(images_dir, image_file),
                os.path.join(output_dir, "images", split_name, image_file)
            )

            shutil.copy(
                os.path.join(labels_dir, txt_file),
                os.path.join(output_dir, "labels", split_name, txt_file)
            )

# ---------------- MAIN ----------------
def main():

    json_dir = r"D:\MLops Training\Day 7\Task 17\Video-To-Image"

    labels_output_dir = r"D:\MLops Training\labels"

    images_output_dir = r"D:\MLops Training\images"

    final_dataset_dir = r"D:\MLops Training\YOLODataset"

    os.makedirs(labels_output_dir, exist_ok=True)
    os.makedirs(images_output_dir, exist_ok=True)

    json_files = list(Path(json_dir).glob("*.json"))

    print("Total JSON files:", len(json_files))

    for json_file in json_files:

        convert_labelme_to_yolo(
            str(json_file),
            labels_output_dir,
            images_output_dir
        )

    print("Conversion completed")

    split_dataset(
        images_output_dir,
        labels_output_dir,
        final_dataset_dir
    )

    print("Dataset splitting completed")

# ---------------- RUN ----------------
if __name__ == "__main__":
    main()